# Chapter 4 Practical: Model-Based Collaborative Filtering

This notebook follows `RS_C4_V4.pdf`: Collaborative Filtering - Model-Based.

Learning objectives:
- Understand how model-based CF differs from memory-based CF.
- Represent users and items with latent factors.
- Use matrix factorization to predict missing ratings.
- Compare simple SVD, ALS, and NMF examples.
- Evaluate rating predictions with MAE and RMSE.


In [1]:
import pandas as pd
import numpy as np
from pathlib import Path
from sklearn.decomposition import NMF
from sklearn.metrics import mean_absolute_error, mean_squared_error

DATA_DIRS = [
    Path("data"),
    Path("../data"),
    Path("chapter_04_model_based_collaborative_filtering/data"),
]
GITHUB_DATA_URL = "https://raw.githubusercontent.com/MehrdadJalali-AI/RecommenderSystems/main/chapter_04_model_based_collaborative_filtering/data"

def read_chapter4_csv(filename):
    for data_dir in DATA_DIRS:
        csv_path = data_dir / filename
        if csv_path.exists():
            print(f"Loaded {filename} from {csv_path}")
            return pd.read_csv(csv_path)
    url = f"{GITHUB_DATA_URL}/{filename}"
    print(f"Local file not found. Loading {filename} from GitHub raw URL.")
    return pd.read_csv(url)

ratings = read_chapter4_csv("ratings_chapter4.csv")
movies = read_chapter4_csv("movies_chapter4.csv")
ratings_named = ratings.merge(movies, on="movie_id", how="left")
rating_matrix = ratings_named.pivot_table(index="user_id", columns="title", values="rating")
rating_matrix


Loaded ratings_chapter4.csv from data/ratings_chapter4.csv
Loaded movies_chapter4.csv from data/movies_chapter4.csv


title,Action Hero,Funny Days,Love Story,Mystery Night,Robot Future,Space Journey
user_id,,,,,,
Anna,5.0,3.0,2.0,NaN,NaN,NaN
Ben,NaN,4.0,4.0,NaN,5.0,5.0
Liam,2.0,5.0,5.0,NaN,NaN,4.0
Mia,5.0,NaN,NaN,5.0,4.0,4.0
Omar,NaN,4.0,5.0,2.0,NaN,2.0
Sara,4.0,NaN,1.0,4.0,4.0,NaN


## Packages and key functions used

- `pandas` is used for tables: `read_csv`, `merge`, `pivot_table`, `groupby`, `sort_values`, and `dropna`.
- `numpy` is used for matrix operations: `dot`, `linalg.svd`, `sqrt`, `clip`, random initialization, and array indexing.
- `sklearn.decomposition.NMF` fits a non-negative matrix factorization model.
- `sklearn.metrics.mean_absolute_error` and `mean_squared_error` evaluate rating prediction error.
- `Path` helps the notebook find CSV files locally or load them from GitHub when opened in Colab.


## Part 1 - From missing ratings to a learned model

Model-based collaborative filtering does not search for nearest neighbors every time. Instead, it learns compact user and item profiles from the known ratings.

In the matrix above:
- rows are users,
- columns are movies,
- numbers are observed ratings,
- missing cells are unknown preferences that we want to predict.


In [2]:
n_users, n_items = rating_matrix.shape
n_known = rating_matrix.notna().sum().sum()
density = n_known / (n_users * n_items)

pd.DataFrame({
    "measure": ["users", "items", "known ratings", "possible ratings", "density", "sparsity"],
    "value": [n_users, n_items, n_known, n_users * n_items, round(density, 3), round(1 - density, 3)],
})


,measure,value
0,users,6.000
1,items,6.000
2,known ratings,23.000
3,possible ratings,36.000
4,density,0.639
5,sparsity,0.361


## Part 2 - Latent factors and dot products

Latent factors are hidden preference dimensions learned from ratings. In the lecture, possible interpretations include Action, Romance, Classic, Modern, Popular, or Specialized.

The names below are only for teaching. In real matrix factorization, the model learns numerical factors automatically and they may not have clear human names.

The dot product estimates a match score:

`score(user, item) = user_factors dot item_factors`


In [3]:
manual_user_factors = pd.DataFrame(
    [[0.9, 0.2], [0.2, 0.9]],
    index=["Anna", "Ben"],
    columns=["Action factor", "Romance factor"],
)

manual_item_factors = pd.DataFrame(
    [[0.8, 0.1], [0.1, 0.8], [0.7, 0.3]],
    index=["Action Hero", "Love Story", "Space Journey"],
    columns=["Action factor", "Romance factor"],
)

match_scores = manual_user_factors.dot(manual_item_factors.T)
match_scores.round(2)


,Action Hero,Love Story,Space Journey
Anna,0.74,0.25,0.69
Ben,0.25,0.74,0.41


## Part 3 - SVD-style matrix factorization

Singular Value Decomposition (SVD) decomposes a matrix into compact latent dimensions. A full recommender would learn from only observed ratings. For this small classroom example, we first fill missing values with user means so `np.linalg.svd()` can work on a complete matrix.

Important functions:
- `fillna()` creates a complete teaching matrix.
- `np.linalg.svd()` factorizes the matrix.
- `k_factors` controls how many latent dimensions are kept.
- `np.clip()` keeps predictions inside the 1-5 rating range.


In [4]:
filled_matrix = rating_matrix.apply(lambda row: row.fillna(row.mean()), axis=1)
matrix_values = filled_matrix.values

U, singular_values, Vt = np.linalg.svd(matrix_values, full_matrices=False)

k_factors = 2
U_k = U[:, :k_factors]
S_k = np.diag(singular_values[:k_factors])
Vt_k = Vt[:k_factors, :]

svd_prediction_values = U_k @ S_k @ Vt_k
svd_predictions = pd.DataFrame(
    np.clip(svd_prediction_values, 1, 5),
    index=rating_matrix.index,
    columns=rating_matrix.columns,
)

svd_predictions.round(2)


title,Action Hero,Funny Days,Love Story,Mystery Night,Robot Future,Space Journey
user_id,,,,,,
Anna,4.34,2.98,1.79,3.80,3.62,3.47
Ben,4.84,4.53,3.91,4.64,4.66,4.36
Liam,2.99,4.70,5.00,3.49,3.88,3.46
Mia,4.65,4.64,4.22,4.55,4.63,4.30
Omar,2.25,3.93,4.74,2.75,3.12,2.76
Sara,4.47,2.79,1.40,3.83,3.58,3.47


The next cell recommends unseen movies for one target user by ranking the predicted ratings for missing entries only.


In [5]:
target_user = "Anna"
seen_items = rating_matrix.loc[target_user].dropna().index
svd_recommendations = (
    svd_predictions.loc[target_user]
    .drop(labels=seen_items)
    .sort_values(ascending=False)
    .rename("predicted_rating")
    .to_frame()
)
svd_recommendations.round(2)


,predicted_rating
title,
Mystery Night,3.80
Robot Future,3.62
Space Journey,3.47


## Part 4 - ALS from observed ratings

Alternating Least Squares (ALS) alternates between two steps:

1. keep item factors fixed and update user factors,
2. keep user factors fixed and update item factors.

This simple implementation uses only observed ratings. It is small enough for students to inspect, but it mirrors the main lecture idea.


In [6]:
def fit_als(matrix, n_factors=2, n_iterations=20, regularization=0.1, random_state=42):
    rng = np.random.default_rng(random_state)
    rating_values = matrix.values.astype(float)
    observed = ~np.isnan(rating_values)
    n_users, n_items = rating_values.shape

    user_factors = rng.normal(0, 0.1, size=(n_users, n_factors))
    item_factors = rng.normal(0, 0.1, size=(n_items, n_factors))
    identity = np.eye(n_factors)

    for _ in range(n_iterations):
        for u in range(n_users):
            item_ids = np.where(observed[u])[0]
            if len(item_ids) == 0:
                continue
            V = item_factors[item_ids]
            r = rating_values[u, item_ids]
            user_factors[u] = np.linalg.solve(V.T @ V + regularization * identity, V.T @ r)

        for i in range(n_items):
            user_ids = np.where(observed[:, i])[0]
            if len(user_ids) == 0:
                continue
            U_obs = user_factors[user_ids]
            r = rating_values[user_ids, i]
            item_factors[i] = np.linalg.solve(U_obs.T @ U_obs + regularization * identity, U_obs.T @ r)

    predictions = user_factors @ item_factors.T
    predictions = np.clip(predictions, 1, 5)
    return user_factors, item_factors, pd.DataFrame(predictions, index=matrix.index, columns=matrix.columns)

als_user_factors, als_item_factors, als_predictions = fit_als(rating_matrix, n_factors=2)
als_predictions.round(2)


title,Action Hero,Funny Days,Love Story,Mystery Night,Robot Future,Space Journey
user_id,,,,,,
Anna,4.89,2.92,2.01,5.00,4.91,4.79
Ben,3.86,4.21,3.88,5.00,5.00,4.84
Liam,2.05,4.80,5.00,4.15,4.23,3.99
Mia,5.00,1.53,1.00,4.80,4.15,4.09
Omar,1.00,4.04,4.90,1.96,2.32,2.11
Sara,4.02,1.89,1.04,4.20,3.73,3.65


The factor matrices are the learned user and item profiles. They are reusable: once learned, the recommender can predict many missing ratings quickly.


In [7]:
user_factor_table = pd.DataFrame(
    als_user_factors,
    index=rating_matrix.index,
    columns=["latent_factor_1", "latent_factor_2"],
)

item_factor_table = pd.DataFrame(
    als_item_factors,
    index=rating_matrix.columns,
    columns=["latent_factor_1", "latent_factor_2"],
)

print("User latent factors:")
display(user_factor_table.round(2))
print("Item latent factors:")
display(item_factor_table.round(2))


User latent factors:


,latent_factor_1,latent_factor_2
user_id,,
Anna,3.23,1.50
Ben,3.82,0.38
Liam,3.72,-0.87
Mia,2.35,2.16
Omar,2.62,-1.81
Sara,2.32,1.45


Item latent factors:


,latent_factor_1,latent_factor_2
title,,
Action Hero,0.87,1.38
Funny Days,1.16,-0.56
Love Story,1.12,-1.08
Mystery Night,1.30,0.80
Robot Future,1.27,0.55
Space Journey,1.21,0.58


## Part 5 - NMF with non-negative factors

Non-Negative Matrix Factorization (NMF) learns only zero or positive factor values. This can make factors easier to interpret because profiles are combined additively.

`sklearn.decomposition.NMF` needs a complete non-negative matrix. For this classroom example, missing values are filled with item means. This filling is only a teaching step, not a claim that missing ratings are real ratings.


In [8]:
nmf_input = rating_matrix.apply(lambda col: col.fillna(col.mean()), axis=0)

nmf_model = NMF(n_components=2, init="random", random_state=7, max_iter=1000)
nmf_user_factors = nmf_model.fit_transform(nmf_input)
nmf_item_factors = nmf_model.components_

nmf_prediction_values = nmf_user_factors @ nmf_item_factors
nmf_predictions = pd.DataFrame(
    np.clip(nmf_prediction_values, 1, 5),
    index=rating_matrix.index,
    columns=rating_matrix.columns,
)

nmf_predictions.round(2)


title,Action Hero,Funny Days,Love Story,Mystery Night,Robot Future,Space Journey
user_id,,,,,,
Anna,4.51,3.42,1.85,4.06,4.02,3.92
Ben,4.30,4.46,3.98,4.00,4.79,4.14
Liam,3.12,4.55,5.00,3.03,4.52,3.43
Mia,4.75,4.17,2.99,4.34,4.68,4.32
Omar,2.73,4.09,4.85,2.67,4.05,3.04
Sara,4.47,3.21,1.49,4.01,3.83,3.83


## Part 6 - Choosing k with validation error

The number of latent factors `k` controls model complexity.

- small `k`: simple model, may underfit,
- large `k`: flexible model, may overfit,
- balanced `k`: chosen using validation error such as RMSE.

The next example hides one rating per user, trains ALS on the remaining ratings, and evaluates MAE/RMSE on the hidden ratings.


In [9]:
test_rows = ratings_named.groupby("user_id", group_keys=False).sample(n=1, random_state=11)
train_rows = ratings_named.drop(test_rows.index)
train_matrix = train_rows.pivot_table(index="user_id", columns="title", values="rating")
train_matrix = train_matrix.reindex(index=rating_matrix.index, columns=rating_matrix.columns)

results = []
for k in [1, 2, 3]:
    _, _, train_predictions = fit_als(train_matrix, n_factors=k, n_iterations=30, regularization=0.2, random_state=7)
    actual = []
    predicted = []
    for _, row in test_rows.iterrows():
        user = row["user_id"]
        title = row["title"]
        actual.append(row["rating"])
        predicted.append(train_predictions.loc[user, title])
    rmse = np.sqrt(mean_squared_error(actual, predicted))
    mae = mean_absolute_error(actual, predicted)
    results.append({"k_factors": k, "MAE": mae, "RMSE": rmse})

pd.DataFrame(results).round(3)


,k_factors,MAE,RMSE
0,1,2.232,2.435
1,2,2.014,2.648
2,3,1.563,2.097


# Challenges

### Challenge 1 - Change the number of latent factors

**Goal:**
Investigate how `k_factors` changes SVD predictions and recommendations.

**What to do:**

1. In the SVD section, change `k_factors = 2` to `k_factors = 1`.
2. Rerun the SVD prediction and recommendation cells.
3. Then try `k_factors = 3`.
4. Compare Anna's recommendation list for the different values of `k_factors`.


In [10]:
# Challenge 1
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Which value of `k_factors` changed the recommendations most? Did a larger `k` always look better? Why can too small or too large `k` be a problem?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 2 - Compare ALS and NMF predictions

**Goal:**
Compare two model-based approaches on the same target user.

**What to do:**

1. Choose one `target_user`, such as `"Anna"` or `"Ben"`.
2. Rank unseen movies for that user using `als_predictions`.
3. Rank unseen movies for the same user using `nmf_predictions`.
4. Compare whether ALS and NMF recommend the same top movie.


In [11]:
# Challenge 2
# Modify or extend the code as described above.

# Write your code below:


### Your observations

Did ALS and NMF choose the same top recommendation? Which prediction table looked easier to interpret? Why might different factorization methods produce different rankings?

> Write your observations here:
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................


### Challenge 3 - Concept Check: Model-Based CF

This challenge requires no programming.

Memory-based CF finds similar users or items directly from the rating matrix. Model-based CF learns latent user and item profiles first.

Explain in your own words:

1. Why can learned latent factors help when the rating matrix is sparse?
2. Why do we need validation error when choosing the number of latent factors?
3. Why might NMF factors be easier to interpret than factors with negative values?

### Your explanation

> ................................................................................
>
> ................................................................................
>
> ................................................................................
>
> ................................................................................
